# Results preparation
Idea here is to take images and compute results per image in self-similar CSV files to run statistics later 

In [48]:
import cv2
import numpy as np
import pandas as pd
import os
import sys
sys.path.append('D:\\TUW\\NecrosisDetector')
from Tools.tools_ML import printProgressBar


In [15]:
def compare_images(pred : np.ndarray, tst : np.ndarray) -> float:
    '''Compare two images using template matching and return the similarity score.
    
    Args:
        pred (np.ndarray): The predicted image.
        tst (np.ndarray): The test image.  
    Returns:
        float: The similarity score between the two images.
    '''
    if np.unique(tst).size == 2:  # If the test image is binary
        #print("Using Jaccard similarity for binary images.")
        return np.logical_and(pred, tst).sum() / np.logical_or(pred, tst).sum()
    else:
        return cv2.matchTemplate(
                    pred.astype(np.uint8),
                    tst.astype(np.uint8),
                    method=cv2.TM_CCOEFF_NORMED,
                )[0][0]

In [65]:
compare_images(pred[30], test[30])

Using Jaccard similarity for binary images.


0.01171875

In [73]:
np.unique(test[1], return_counts=True)

(array([  0,   1,   2,   3,   4,   5,   6,   7, 249, 250, 251, 252, 253,
        254, 255], dtype=uint8),
 array([3812352,   16320,    9600,    5056,    1408,     704,     128,
            128,     192,     512,    1536,    4352,    8128,   14912,
         318976], dtype=int64))

In [12]:
def read_images (pred_path : str, tst_path : str) -> tuple[list[np.ndarray], list[np.ndarray], list[str]]:
    '''Read two images from the given paths and return them as numpy arrays.
    
    Args:
        pred_path (str): The path to the folder where predicted images are.
        tst_path (str): The path to the folder where test images are.
    Returns:
        tuple[list[np.ndarray], list[np.ndarray], list[str]]: A tuple containing lists of predicted images and test images as numpy arrays, and a list of filenames for the test images.
    '''
    pred_images = []
    tst_images = []
    filenames = []    
    
    for filename in os.listdir(pred_path):
        if filename.endswith((".png")):
            # predicted image
            #print(f"Reading {filename}...")
            img = cv2.imread(os.path.join(pred_path, filename), cv2.IMREAD_GRAYSCALE)
            pred_images.append(img)
            pred_shape = img.shape
            # test
            img = cv2.imread(os.path.join(tst_path, filename), cv2.IMREAD_GRAYSCALE)
            #print(img.shape)
            img = cv2.resize(img, pred_shape, interpolation=cv2.INTER_NEAREST)
            tst_images.append(img)
            # list of filenames for the test images 
            filenames.append(filename)
    
    return pred_images, tst_images, filenames

In [24]:
def read_folder ( path : str) -> tuple[list[np.ndarray], list[str]]:
    '''Read images from the given path and return them as numpy arrays.
    
    Args:
        path (str): The path to the folder where images are.
    Returns:
        tuple[list[np.ndarray], list[str]]: A tuple containing lists of images as numpy arrays and a list of filenames.
    '''
    images = []
    filenames = []    
    
    for filename in os.listdir(path):
        # predicted image
        #print(f"Reading {filename}...")
        try:
            img = cv2.imread(os.path.join(path, filename), cv2.IMREAD_GRAYSCALE)
        except TypeError as e:
            print(f"Error reading {filename}: {e}")
            continue
        images.append(img)
        
        # list of filenames for the test images 
        filenames.append(filename)
    
    return images, filenames

In [28]:
pred, test, filenames = read_images(tst_path= r"D:\TUW\Images\ht29\pi_mask", pred_path= r"D:\TUW\Images\ht29\ht29_dapi_results_dt_classifier")
similarity_scores = [compare_images(p, t) for p, t in zip(pred, test)]

Reading a1.png...
(2048, 2048)
Reading a2.png...
(2048, 2048)
Reading a4.png...
(2048, 2048)
Reading a5.png...
(2048, 2048)
Reading a6.png...
(2048, 2048)
Reading a7.png...
(2048, 2048)
Reading a8.png...
(2048, 2048)
Reading a9.png...
(2048, 2048)
Reading b1.png...
(2048, 2048)
Reading b2.png...
(2048, 2048)
Reading b4.png...
(2048, 2048)
Reading b5.png...
(2048, 2048)
Reading b6.png...
(2048, 2048)
Reading b7.png...
(2048, 2048)
Reading b8.png...
(2048, 2048)
Reading b9.png...
(2048, 2048)
Reading c1.png...
(2048, 2048)
Reading c2.png...
(2048, 2048)
Reading c4.png...
(2048, 2048)
Reading c5.png...
(2048, 2048)
Reading c6.png...
(2048, 2048)
Reading d1.png...
(2048, 2048)
Reading d2.png...
(2048, 2048)
Reading d3.png...
(2048, 2048)
Reading d4.png...
(2048, 2048)
Reading d5.png...
(2048, 2048)
Reading d6.png...
(2048, 2048)
Reading d7.png...
(2048, 2048)
Reading d9.png...
(2048, 2048)
Reading e1.png...
(2048, 2048)
Reading e10.png...
(2048, 2048)
Reading e2.png...
(2048, 2048)
Reading

In [51]:
cell_lines = ["ht29", "pancreas", "both"]
dyes = ["pi", "dapi"]
models = ["otsu_equalize", "watershed", "water_means"]

results = pd.DataFrame()
for model in models:
    for dye in dyes:
        for cell_line in cell_lines:
            print(f"Processing {model} for {cell_line} with {dye} dye...")      
            pred, test, filenames = read_images(
                tst_path=f"D:/TUW/Images/{cell_line}/{dye}_mask",
                #tst_path=f"D:/TUW/Images/{cell_line}/{dye}",
                #pred_path=f"D:/TUW/Images/{cell_line}/{cell_line}_{dye}_results_{model}")
                pred_path=f"D:/TUW/Images/{cell_line}/results_{model}")
        
            col_name = f"{cell_line}-{dye}"
            for pred, test, filename in zip(pred, test, filenames):
                results.loc[filename.replace(".png", ""), col_name] = compare_images(pred, test)


            results.to_csv(f"../Results/similarity_scores_{model}.csv", index=True)

Processing ht29 with pi dye...
Processing pancreas with pi dye...
Processing both with pi dye...
Processing ht29 with dapi dye...
Processing pancreas with dapi dye...
Processing both with dapi dye...
Processing ht29 with pi dye...
Processing pancreas with pi dye...
Processing both with pi dye...
Processing ht29 with dapi dye...
Processing pancreas with dapi dye...
Processing both with dapi dye...
Processing ht29 with pi dye...
Processing pancreas with pi dye...
Processing both with pi dye...
Processing ht29 with dapi dye...
Processing pancreas with dapi dye...
Processing both with dapi dye...


In [31]:
os.listdir(f"D:/TUW/Images/{cell_line}/{cell_line}_{dye}_results_{model}_classifier")

['a1.jpg',
 'a1.png',
 'a2.jpg',
 'a2.png',
 'a4.jpg',
 'a4.png',
 'a5.jpg',
 'a5.png',
 'a6.jpg',
 'a6.png',
 'a7.jpg',
 'a7.png',
 'a8.jpg',
 'a8.png',
 'a9.jpg',
 'a9.png',
 'b1.jpg',
 'b1.png',
 'b2.jpg',
 'b2.png',
 'b4.jpg',
 'b4.png',
 'b5.jpg',
 'b5.png',
 'b6.jpg',
 'b6.png',
 'b7.jpg',
 'b7.png',
 'b8.jpg',
 'b8.png',
 'b9.jpg',
 'b9.png',
 'c1.jpg',
 'c1.png',
 'c2.jpg',
 'c2.png',
 'c4.jpg',
 'c4.png',
 'c5.jpg',
 'c5.png',
 'c6.jpg',
 'c6.png',
 'd1.jpg',
 'd1.png',
 'd2.jpg',
 'd2.png',
 'd3.jpg',
 'd3.png',
 'd4.jpg',
 'd4.png',
 'd5.jpg',
 'd5.png',
 'd6.jpg',
 'd6.png',
 'd7.jpg',
 'd7.png',
 'd9.jpg',
 'd9.png',
 'e1.jpg',
 'e1.png',
 'e10.jpg',
 'e10.png',
 'e2.jpg',
 'e2.png',
 'e3.jpg',
 'e3.png',
 'e4.jpg',
 'e4.png',
 'e5.jpg',
 'e5.png',
 'e6.jpg',
 'e6.png',
 'e7.jpg',
 'e7.png',
 'e8.jpg',
 'e8.png',
 'e9.jpg',
 'e9.png',
 'f1.jpg',
 'f1.png',
 'f10.jpg',
 'f10.png',
 'f2.jpg',
 'f2.png',
 'f3.jpg',
 'f3.png',
 'f4.jpg',
 'f4.png',
 'f5.jpg',
 'f5.png',
 'f6.j

# Generate Datasets

#### Function definition

In [7]:
def otsu_equalize (bf) :
  # take the negative of the image
  bf_neg = 255 - bf

  bf_neg = cv2.GaussianBlur(bf_neg, (5,5),0)

  (T, threshInv) = cv2.threshold(bf_neg, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)
  threshInv = cv2.bitwise_and(bf_neg, bf_neg, mask=threshInv)

  # increase the contrast
  alpha = 0.7
  threshInv = np.clip(alpha*threshInv, 0, 255).astype(np.uint8)
  threshInv = cv2.equalizeHist(threshInv) # now equalize the histogram that has got narrowed by a factor alpha

  (T, bf_final) = cv2.threshold(threshInv, 100 , 255, cv2.THRESH_BINARY)
  #(T, bf_final) = cv2.threshold(bf_neg, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)

  return bf_final

def kmeans (bf, K = 2) :
  # take the negative of the image
  #bf_neg = 255 - bf
  bf_neg = bf

  z = np.float32(bf_neg.reshape((-1,1)))
  criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 5, 1.0)
  compactness,labels,center = cv2.kmeans(z , K, None, criteria, 5, cv2.KMEANS_RANDOM_CENTERS)
  center = np.uint8(center)
  res = center[labels.flatten()]
  bf_final = res.reshape((bf_neg.shape))

  #bf_final = cv2.threshold(bf_final, 150, 255, cv2.THRESH_BINARY)[1]

  return bf_final

def watershed(bf):

  bf_neg = 255 - bf
  (T, threshInv) = cv2.threshold(bf_neg, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)
  #threshInv = cv2.bitwise_and(img[i], img[i], mask=threshInv)


  kernel = np.ones((3,3),np.uint8)
  opening = cv2.morphologyEx(threshInv,cv2.MORPH_OPEN,kernel, iterations = 2)
  opening = cv2.dilate(opening,kernel, iterations = 2) # probably results improve when we play with this

  # Finding sure foreground area
  dist_transform = cv2.distanceTransform(opening,cv2.DIST_L2,3) # probably results improve when we play with this
  ret, sure_fg = cv2.threshold(dist_transform,0.5*dist_transform.max(),255,0) # probably results improve when we play with this
  # Finding unknown region
  sure_fg = np.uint8(sure_fg)
  unknown = cv2.subtract(opening,sure_fg)

  # Marker labelling
  ret, markers = cv2.connectedComponents(sure_fg)
  # Add one to all labels so that sure background is not 0, but 1
  markers = markers+1
  # Now, mark the region of unknown with zero
  markers[unknown==255] = 0


  bf_neg = cv2.cvtColor(bf_neg,cv2.COLOR_GRAY2RGB)
  markers = cv2.watershed(bf_neg,markers)


  res = 255*(markers == 2)

  #res = np.logical_and(bf_neg, res)

  return res

def water_means(img):
  w = watershed(img).astype(np.uint8)
  wi = cv2.bitwise_and(255-img, 255-img, mask = w)
  ki = kmeans(wi, K = 3)
  return 255*(ki == np.max(ki))

#### Generate images in Loop

In [50]:
results = pd.DataFrame()
for method in otsu_equalize, watershed, water_means:
    for cell_line in ["ht29", "pancreas", "both"]:
        #check if folder exists, if not create it
        if not os.path.exists(f"D:/TUW/Images/{cell_line}/results_{method.__name__}"):
            os.makedirs(f"D:/TUW/Images/{cell_line}/results_{method.__name__}")


        print(f"Processing {cell_line} using {method.__name__} method...")      
        brightfield, filenames = read_folder(
            path=f"D:/TUW/Images/{cell_line}/brightfield")

        # save processed images in corresponding folders
        for bf, filename in zip(brightfield, filenames):
            processed = method(bf)
            printProgressBar(filenames.index(filename) +1 , len(filenames), prefix = 'Progress:', suffix = '', length = 50)
            filename = filename.replace("jpg", "png")
            
            cv2.imwrite(f"D:/TUW/Images/{cell_line}/results_{method.__name__}/{filename}", processed)


Processing ht29 using otsu_equalize method...
Progress: |██████████████████████████████████████████████████| 100.0% 
Processing pancreas using otsu_equalize method...
Progress: |██████████████████████████████████████████████████| 100.0% 
Processing both using otsu_equalize method...
Progress: |██████████████████████████████████████████████████| 100.0% 
Processing ht29 using watershed method...
Progress: |██████████████████████████████████████████████████| 100.0% 
Processing pancreas using watershed method...
Progress: |██████████████████████████████████████████████████| 100.0% 
Processing both using watershed method...
Progress: |██████████████████████████████████████████████████| 100.0% 
Processing ht29 using water_means method...
Progress: |██████████████████████████████████████████████████| 100.0% 
Processing pancreas using water_means method...
Progress: |██████████████████████████████████████████████████| 100.0% 
Processing both using water_means method...
Progress: |█████████████

In [21]:
test

[]

# Additional tools

#### JPG to PNG

In [12]:
def maskJPGtoPNG(folder : str, verbose : bool = False) -> None:
    for filename in os.listdir(folder):
        img = cv2.imread(os.path.join(folder, filename), cv2.IMREAD_GRAYSCALE)
        #_, img = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)
        cv2.imwrite(os.path.join(folder, filename.replace(".jpg", ".png")), img)
        if verbose:
            print(np.unique(img, return_counts=True))

In [13]:
maskJPGtoPNG(r"D:\TUW\Images\ht29\dapi", verbose=False)
maskJPGtoPNG(r"D:\TUW\Images\pancreas\dapi", verbose=False)
maskJPGtoPNG(r"D:\TUW\Images\both\dapi", verbose=False)
maskJPGtoPNG(r"D:\TUW\Images\ht29\pi", verbose=False)
maskJPGtoPNG(r"D:\TUW\Images\pancreas\pi", verbose=False)
maskJPGtoPNG(r"D:\TUW\Images\both\pi", verbose=False)

#### Merge column to DF

In [4]:
extra_col = pd.read_csv("../Results/UNET_cont_both_pi.csv", index_col=0)
results_col = pd.read_csv("../Results/results_UNET_cont.csv", index_col=0)

In [5]:
results_col["both-pi"] = extra_col["both-pi"]
results_col.to_csv("../Results/results_UNET_cont.csv", index=True)